In [ ]:
# Credit Risk Assessment (Credit Card Default)

Goal: build a model that predicts whether a customer will default on their credit card (`credit_card_default`).

This notebook:
- loads `train.csv` / `test.csv`
- performs quick EDA
- builds a robust preprocessing + modeling pipeline
- evaluates on a validation split (ROC-AUC, PR-AUC, confusion matrix)
- generates `submission_predictions.csv` for the test set


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_DIR = Path(".")
TRAIN_PATH = PROJECT_DIR / "train.csv"
TEST_PATH = PROJECT_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("train:", train.shape)
print("test:", test.shape)
train.head()

In [ ]:
TARGET = "credit_card_default"
ID_COL = "customer_id"

assert TARGET in train.columns
assert TARGET not in test.columns

# Basic checks
print(train[TARGET].value_counts(dropna=False))
print("Positive rate:", train[TARGET].mean())

missing_pct = (train.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(12)

In [ ]:
# EDA: target distribution
plt.figure(figsize=(4, 3))
sns.countplot(x=TARGET, data=train)
plt.title("Target distribution")
plt.tight_layout()
plt.show()

# Numeric feature distributions (a few key fields)
num_cols = [
    "age",
    "net_yearly_income",
    "no_of_days_employed",
    "yearly_debt_payments",
    "credit_limit",
    "credit_limit_used(%)",
    "credit_score",
    "prev_defaults",
    "default_in_last_6months",
]
num_cols = [c for c in num_cols if c in train.columns]

train[num_cols].describe().T

In [ ]:
# Quick relationships: credit score vs default
if "credit_score" in train.columns:
    plt.figure(figsize=(6, 4))
    sns.kdeplot(data=train, x="credit_score", hue=TARGET, common_norm=False)
    plt.title("Credit score distribution by default")
    plt.tight_layout()
    plt.show()

# Correlation heatmap (numeric)
num_train = train.select_dtypes(include="number")
plt.figure(figsize=(10, 7))
sns.heatmap(num_train.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation heatmap (numeric features)")
plt.tight_layout()
plt.show()

In [ ]:
# Modeling

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = train.drop(columns=[TARGET])
y = train[TARGET].astype(int)

# drop obvious leakage/ID-like columns
drop_cols = [c for c in [ID_COL, "name"] if c in X.columns]
X = X.drop(columns=drop_cols)
X_test_final = test.drop(columns=drop_cols)

categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                n_jobs=None,
            ),
        ),
    ]
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

val_proba = model.predict_proba(X_val)[:, 1]
val_pred = (val_proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, val_proba))
print("PR-AUC:", average_precision_score(y_val, val_proba))
print("\nClassification report (threshold=0.5):\n")
print(classification_report(y_val, val_pred))

cm = confusion_matrix(y_val, val_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix (threshold=0.5)")
plt.xlabel("Pred")
plt.ylabel("True")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_val, val_proba)
plt.show()
PrecisionRecallDisplay.from_predictions(y_val, val_proba)
plt.show()

In [ ]:
# Feature impact (Logistic Regression coefficients)

import numpy as np

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_.ravel()

coef_df = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

coef_df.head(20)

In [ ]:
# Train on full training data, generate predictions for test set

model.fit(X, y)

test_proba = model.predict_proba(X_test_final)[:, 1]

pred_path = PROJECT_DIR / "submission_predictions.csv"

out = pd.DataFrame({
    ID_COL: test[ID_COL] if ID_COL in test.columns else np.arange(len(test_proba)),
    "credit_card_default_probability": test_proba,
    "credit_card_default_pred_0_5": (test_proba >= 0.5).astype(int),
})

out.to_csv(pred_path, index=False)
print("Wrote:", pred_path)
out.head()